# Memory and surgery circuits

Build memory and surgery circuits (phenomenological noise) and find the minimum-weight faults that cause a logical error.

In [1]:
import galois
import networkx as nx
from sympy import symbols, Poly

from core.CSS import LPCode, RotatedSurfaceCode
from core.codes import LPCodePatch, RotatedSurfaceCodePatch, AncillaGraph
from surgery.circuit import CircuitBuilder
from surgery.surgery import MergedCodeWithGraph
from surgery.phenomenological_circuit import MemoryCircuitScheduler, SurgeryCircuitScheduler
from analysis.logical_fault_analyzer import ErrorSimulator, LogicalFaultAnalyzer, SurgeryLogicalFaultAnalyzer

## LP code memory circuit

In [2]:
x = symbols('x')
ring = Poly(x**3+1, x, modulus=2)
Ha = [[Poly(1+x**2,x, modulus=2),Poly(x+x**2,x, modulus=2)],
        [Poly(1+x,x, modulus=2),Poly(x+x**2,x, modulus=2)]]
Hb = [[Poly(1+x**2,x, modulus=2),Poly(x+x**2,x, modulus=2)],
        [Poly(1+x,x, modulus=2),Poly(1+x,x, modulus=2)]]
lp1 = LPCode(Ha=Ha, Hb=Hb, R=ring)

op = [[0]*24 for _ in range(8)]
for i in range(8):
    op[i][3*i:3*i+3] = [1]*3
lp1.set_logical_representatives(op, op)

patch = LPCodePatch(css=lp1, patch_name="lp_patch1")

scheduler = MemoryCircuitScheduler(patch, rounds=3, p_data=0.01, p_meas=0.01)
schedule = scheduler.build()

cb = CircuitBuilder(schedule)
cb.build()

# print(cb.circuit)
# print()
# for idx, record in cb.meas_record.items():
#     check_label = record['check'].label if record['check'] is not None else None
#     print(f"  [{idx}]: tick={record['tick']}, check={check_label}, label={record['label']}")

## Minimum fault weight for each logical X

In [3]:
scheduler = MemoryCircuitScheduler(patch, rounds=3, p_data=0.01, p_meas=0.01)
schedule = scheduler.build()

cb = CircuitBuilder(schedule)
cb.build()
cb.add_detectors()
circ = cb.circuit

analyzer = LogicalFaultAnalyzer(ErrorSimulator(circ), patch)

# find the min fault that flips logical X_i
for i in range(8):
    print(f"\n=== min fault flipping logical X_{i} ===")
    e = analyzer.find_min_weight(error_type='Z', observable_indices=[i], verbose=False)
    min_wt = int(e.sum())
    print(min_wt)


=== min fault flipping logical X_0 ===


3

=== min fault flipping logical X_1 ===


3

=== min fault flipping logical X_2 ===


3

=== min fault flipping logical X_3 ===


3

=== min fault flipping logical X_4 ===


3

=== min fault flipping logical X_5 ===


3

=== min fault flipping logical X_6 ===


3

=== min fault flipping logical X_7 ===


3


## [[40,8,5]] LP code

In [4]:
x = symbols('x')
ring = Poly(x**5+1, x, modulus=2)
Ha = [[Poly(x+x**3,x,modulus=2),Poly(x+x**4,x,modulus=2)],
      [Poly(x+x**4,x,modulus=2),Poly(x+x**4,x,modulus=2)]]
Hb = [[Poly(1+x,x,modulus=2),Poly(x**3+x**4,x,modulus=2)],
      [Poly(1+x**4,x,modulus=2),Poly(1+x**4,x,modulus=2)]]
lp2 = LPCode(Ha=Ha, Hb=Hb, R=ring)
op = [[0]*40 for _ in range(8)]
for i in range(8): op[i][5*i:5*i+5] = [1]*5
lp2.set_logical_representatives(op, op)
# print(lp2.logical_x_repr)
patch = LPCodePatch(css=lp2, patch_name='lp_patch2')

scheduler = MemoryCircuitScheduler(patch, rounds=1, p_data=0.01, p_meas=0.01)
schedule = scheduler.build()

cb = CircuitBuilder(schedule)
cb.build()
cb.add_detectors()
circ = cb.circuit
# print(circ)

analyzer = LogicalFaultAnalyzer(ErrorSimulator(circ), patch)
for i in range(8):
    print(f"\n=== min fault flipping logical X_{i} ===")
    e = analyzer.find_min_weight(error_type='Z', observable_indices=[i], verbose=False)
    min_wt = int(e.sum())
    print(min_wt)


=== min fault flipping logical X_0 ===


5

=== min fault flipping logical X_1 ===


5

=== min fault flipping logical X_2 ===


5

=== min fault flipping logical X_3 ===


5

=== min fault flipping logical X_4 ===


5

=== min fault flipping logical X_5 ===


5

=== min fault flipping logical X_6 ===


5

=== min fault flipping logical X_7 ===


5


## Rotated surface code memory circuit

In [5]:
d=7
surf = RotatedSurfaceCode(d=d)
x0=[0]*(d**2)
for i in range(d):
    x0[i]=1
z0=[0]*(d**2)
for i in range(d):
    z0[d*i]=1
surf.set_logical_representative("X",0,x0)
surf.set_logical_representative("Z",0,z0)
patch = RotatedSurfaceCodePatch(surf)
p=1e-3
scheduler = MemoryCircuitScheduler(patch, rounds=7, p_data=p, p_meas=p)
schedule = scheduler.build()
cb = CircuitBuilder(schedule)
cb.build()
cb.add_detectors()
circ = cb.circuit
# print(circ)
print(f"qubits: {circ.num_qubits}, measurements: {circ.num_measurements}, detectors: {circ.num_detectors}")

qubits: 49, measurements: 433, detectors: 360


## LP code surgery circuit

In [6]:
"""
[[24,8,3]] LP code
"""
x = symbols('x')
ring = Poly(x**3+1, x, modulus=2)
Ha = [[Poly(1+x**2,x, modulus=2),Poly(x+x**2,x, modulus=2)],
        [Poly(1+x,x, modulus=2),Poly(x+x**2,x, modulus=2)]]
Hb = [[Poly(1+x**2,x, modulus=2),Poly(x+x**2,x, modulus=2)],
        [Poly(1+x,x, modulus=2),Poly(1+x,x, modulus=2)]]
lp1 = LPCode(Ha=Ha, Hb=Hb, R=ring)  # [[24,8,3]] code

op = [[0]*24 for _ in range(8)]
for i in range(8):
    op[i][3*i:3*i+3] = [1]*3
lx = op
lz = op
lp1.set_logical_representatives(lx, lz)

lp_patch1 = LPCodePatch(css=lp1, patch_name="lp_patch1", offset=(0.0,0.0))

"""
[[40,8,5]] LP code
"""
x = symbols('x')
ring = Poly(x**5+1, x, modulus=2)

Ha = [[Poly(x+x**3,x, modulus=2),Poly(x+x**4,x, modulus=2)],
        [Poly(x+x**4,x, modulus=2),Poly(x+x**4,x, modulus=2)]]
Hb = [[Poly(1+x,x, modulus=2),Poly(x**3+x**4,x, modulus=2)],
        [Poly(1+x**4,x, modulus=2),Poly(1+x**4,x, modulus=2)]]
lp2 = LPCode(Ha=Ha, Hb=Hb, R=ring)  # [[40,8,5]] code
op = [[0]*40 for _ in range(8)]
for i in range(8):
    op[i][5*i:5*i+5] = [1]*5
lx = op
lz = op
lp2.set_logical_representatives(lx, lz)

lp_patch2 = LPCodePatch(css=lp2, patch_name="lp_patch2", offset=(0.0,0.0))

"""
Ancillary graph
"""
G = nx.Graph()
G.add_edges_from([(0,1),(1,2),(2,3),(3,4),(4,0),(0,2),(0,3)])
anc = AncillaGraph(G, patch_name="ancilla", type="X", offset=(0.0,0.0))
pos = nx.spring_layout(G, seed=42)

"""
Merge
"""
X1 = lp_patch1.get_logical_representative("X", 6)
X2 = lp_patch2.get_logical_representative("X", 1)

anc._offset = (4.0, 5.0)  # move the ancilla graph between the two LP patches
anc.scale = (4.0, 4.0)  # scale up the ancilla graph
lp_patch2._offset = (9.0, 0.0)  # move lp_patch2 to the right
merged_code = MergedCodeWithGraph(lp_patch1, lp_patch2, anc, X1=X1, X2=X2, name="merged_lp_code")
merged_code.make_css()
css = merged_code.css

"""
Set logicals
"""
# Measure X_7 of lp1 times X_2 of lp2
# 15 logicals in this order: X_1,...,X_8 of lp1, then X_1,X_3,...,X_8 of lp2
for idx in range(8):
    support = galois.GF(2)([0]*css.n)
    op = lp_patch1.get_logical_representative("X", idx)
    for qubit in op:
        support[merged_code.get_idx(qubit)] = 1
    merged_code.css.set_logical_representative("X", idx, support)
for i, idx in enumerate([0,2,3,4,5,6,7]):
    support = galois.GF(2)([0]*css.n)
    op = lp_patch2.get_logical_representative("X", idx)
    for qubit in op:
        support[merged_code.get_idx(qubit)] = 1
    merged_code.css.set_logical_representative("X", i+8, support)

# Logical Z
for idx in [0,1,2,3,4,5,7]:
    support = galois.GF(2)([0]*css.n)
    op = lp_patch1.get_logical_representative("Z", idx)
    for qubit in op:
        support[merged_code.get_idx(qubit)] = 1
    merged_code.css.set_logical_representative("Z", idx, support)
for i, idx in enumerate([0,2,3,4,5,6,7]):
    support = galois.GF(2)([0]*css.n)
    op = lp_patch2.get_logical_representative("Z", idx)
    for qubit in op:
        support[merged_code.get_idx(qubit)] = 1
    merged_code.css.set_logical_representative("Z", i+8, support)
merged_code.css.set_missing_logicals()

"""
Surgery circuit
"""
p = 1e-3
scheduler = SurgeryCircuitScheduler(merged_code, 3, 5, 3, p, p)
schedule = scheduler.build()
cb = CircuitBuilder(schedule)
cb.build()
cb.add_detectors()
circ = cb.circuit

error_sim = ErrorSimulator(circ)
analyzer = SurgeryLogicalFaultAnalyzer(error_sim, merged_code, cb)
print(f"faults: {analyzer.n_faults}, detectors: {analyzer.n_detectors}")
e = analyzer.find_min_weight(verbose=False)
print(f"min fault weight: {int(e.sum())}")

faults: 2229, detectors: 771


min fault weight: 3
